In [ ]:
# ==========================================
# Cell 1: 基础配置 (V3.4.0 - Ward Grouping + Sliding Window + CNN-BiLSTM)
# ==========================================
import os, pathlib, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import find_peaks
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('.')
DATA_DIR = ROOT / 'data' 

OUT_DIR = ROOT / 'output_340'
CM_DIR = OUT_DIR / 'confusion_matrices'
MODEL_DIR = OUT_DIR / 'models'

for d in [CM_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f'Device: {DEVICE}')
print(f'Output Base Dir: {OUT_DIR}')

In [ ]:
# ==========================================
# Cell 2: 基础解析工具与卡尔曼滤波
# ==========================================
stations_list_path = DATA_DIR / 'stations_list'
station_ids = [int(x) for x in stations_list_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]').split() if x.isdigit()]
TOTAL_STATIONS = len(station_ids)

def parse_day_matrix(line: str):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            mat = parse_day_matrix(line)
            yield mat if mat is not None else None

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

labels_train = load_labels(DATA_DIR / 'PEMS_trainlabels')
labels_test  = load_labels(DATA_DIR / 'PEMS_testlabels')

def apply_kalman_filter(curve, process_variance=1e-4, measurement_variance=1e-2):
    n = len(curve)
    if n == 0: return curve
    xhat, P = np.zeros(n), np.zeros(n)
    xhat[0], P[0] = curve[0], 1.0
    for k in range(1, n):
        xhat_minus = xhat[k-1]
        P_minus = P[k-1] + process_variance
        K = P_minus / (P_minus + measurement_variance) 
        xhat[k] = xhat_minus + K * (curve[k] - xhat_minus)
        P[k] = (1 - K) * P_minus
    return xhat

In [ ]:
# ==========================================
# Cell 3: 5维特征提取 + Ward聚类 (生成分组掩码)
# ==========================================
print(">>> 正在遍历训练集以计算工作日/周末均值曲线...")
sum_wd = np.zeros((TOTAL_STATIONS, 144)); cnt_wd = np.zeros(TOTAL_STATIONS)
sum_we = np.zeros((TOTAL_STATIONS, 144)); cnt_we = np.zeros(TOTAL_STATIONS)

train_path = DATA_DIR / 'PEMS_train'
for day_i, mat in enumerate(iter_day_matrices(train_path)):
    if day_i >= len(labels_train) or mat is None or mat.shape[0] != TOTAL_STATIONS or mat.shape[1] < 144: 
        continue
        
    lb = int(labels_train[day_i])  
    trimmed = mat[:TOTAL_STATIONS, :144]
    
    for sid in range(trimmed.shape[0]):
        curve = trimmed[sid]
        valid = ~np.isnan(curve)
        if valid.any() and not valid.all():
            trimmed[sid] = np.interp(np.arange(144), np.where(valid)[0], curve[valid])
            
    trimmed = np.nan_to_num(trimmed, nan=0.0)
    if lb in [1,2,3,4,5]:
        sum_wd += trimmed; cnt_wd += 1
    elif lb in [6,7]:
        sum_we += trimmed; cnt_we += 1

print(">>> 正在提取 5 维形状特征...")
features_list = []
for sid in range(TOTAL_STATIONS):
    curve_wd = sum_wd[sid] / max(1, cnt_wd[sid])
    curve_we = sum_we[sid] / max(1, cnt_we[sid])
    
    peaks_wd, _ = find_peaks(curve_wd, prominence=0.03, width=2)
    n_peaks = min(len(peaks_wd), 3)
    morn_str = np.max(curve_wd[36:55]) / (np.max(curve_wd) + 1e-8) if len(curve_wd)>54 else 0
    eve_str = np.max(curve_wd[96:115]) / (np.max(curve_wd) + 1e-8) if len(curve_wd)>114 else 0
    wd_we_diff = np.clip(np.linalg.norm(curve_wd - curve_we) / np.sqrt(144), 0, 1)
    
    we_slope = np.max(np.abs(np.diff(curve_we[30:51]))) if len(curve_we)>50 else 0
    peaks_we, _ = find_peaks(curve_we, prominence=0.03, width=2)
    we_morn_mean = np.mean(curve_we[30:55]) if len(curve_we)>54 else 0
    anomaly = min([we_slope/0.02 if we_slope>0 else 0, 1 if len(peaks_we)>=2 else 0, we_morn_mean/0.5 if we_morn_mean>0.3 else 0])
    
    features_list.append([n_peaks/3.0, morn_str, eve_str, wd_we_diff, np.clip(anomaly, 0, 1)])

X_features = np.array(features_list, dtype=np.float32)

print(">>> 正在执行 Ward 层次聚类 (K=5)...")
scaler = StandardScaler()
X_std = scaler.fit_transform(X_features)
dist_matrix = pdist(X_std, metric='euclidean')
Z = linkage(dist_matrix, method='ward')
cluster_labels = fcluster(Z, t=5, criterion='maxclust')

# 寻找质心并剔除各组最远的 10% 离群点
distances = np.zeros(TOTAL_STATIONS)
for k in range(1, 6):
    idx = np.where(cluster_labels == k)[0]
    centroid = X_std[idx].mean(axis=0) 
    distances[idx] = np.linalg.norm(X_std[idx] - centroid, axis=1)

threshold_dist = np.percentile(distances, 90)
global_keep_mask = distances <= threshold_dist

group_station_masks = {}
for k in range(1, 6):
    group_station_masks[f'g{k}'] = (cluster_labels == k) & global_keep_mask

print(f"--- 聚类统计 ---")
print(f"总站点数: {TOTAL_STATIONS}, 剔除离群站点: {TOTAL_STATIONS - global_keep_mask.sum()}")
for k in range(1, 6):
    print(f"Group {k} (g{k}) 有效站点数: {group_station_masks[f'g{k}'].sum()}")

In [ ]:
# ==========================================
# Cell 4: 分组掩码 + 滑动窗口 数据增强加载器
# ==========================================
def process_and_slice_curve(curve_144, is_train, window_size=132, stride=2):
    curve = curve_144[:144]
    if curve.shape[0] < 144:
        curve = np.pad(curve, (0, 144 - curve.shape[0]), mode='constant', constant_values=np.nan)
    
    idx = np.arange(curve.size)
    mask = ~np.isnan(curve)
    if mask.any(): curve = np.interp(idx, idx[mask], curve[mask])
    curve = np.nan_to_num(curve, nan=0.0)
    
    curve = apply_kalman_filter(curve)

    slices = []
    if is_train:
        for offset in range(0, 144 - window_size + 1, stride):
            slices.append(curve[offset : offset + window_size])
    else:
        center_offset = (144 - window_size) // 2
        slices.append(curve[center_offset : center_offset + window_size])

    results = []
    for sl in slices:
        min_val, max_val = sl.min(), sl.max()
        denom = max_val - min_val + 1e-8
        norm_curve = (sl - min_val) / denom
        
        mean_val, std_val = np.mean(norm_curve), np.std(norm_curve)
        log_max_val = np.log1p(max_val) / 10.0 
        stats = np.array([mean_val, std_val, log_max_val], dtype=np.float32)

        grad_curve = np.gradient(norm_curve)
        two_channel_seq = np.stack([norm_curve, grad_curve], axis=0).astype(np.float32)
        results.append((two_channel_seq, stats))
        
    return results

class PEMS_GroupSlidingDataset(Dataset):
    def __init__(self, split_path, labels, use_mask, allowed_days=None, is_train=False, window_size=132, stride=2, split_name=""):
        self.samples = []
        kept_station_ids = np.asarray(station_ids)[use_mask]
        kept_station_pos = np.nonzero(use_mask)[0]
        
        for day_i, mat in enumerate(iter_day_matrices(split_path)):
            if day_i >= len(labels): break
            if allowed_days is not None and day_i not in allowed_days: continue
            if mat is None or mat.shape[0] != TOTAL_STATIONS or mat.shape[1] < 144: continue
                
            y = int(labels[day_i]) - 1
            if y < 0 or y > 6: continue

            mat_group = mat[use_mask, :]
            
            for local_sidx in range(mat_group.shape[0]):
                processed_slices = process_and_slice_curve(mat_group[local_sidx, :144], is_train, window_size, stride)
                
                for seq, stats in processed_slices:
                    if not np.isfinite(seq).all(): continue
                    meta = {
                        "station_id": int(kept_station_ids[local_sidx]),
                        "station_pos": int(kept_station_pos[local_sidx]),
                        "day_i": int(day_i), "split": split_name,
                    }
                    self.samples.append((seq, stats, y, meta))
                
        self.n = len(self.samples)
        print(f"[{split_name}] 加载完成: 包含 {self.n} 条样本 (is_train={is_train}).")

    def __len__(self): return self.n
    def __getitem__(self, idx):
        seq, stats, y, meta = self.samples[idx]
        seq_tensor = torch.from_numpy(seq)
        if meta['split'] == 'train_augmented':
            seq_tensor = seq_tensor * random.uniform(0.95, 1.05)
            seq_tensor += torch.randn_like(seq_tensor) * 0.005
        return seq_tensor, torch.from_numpy(stats), torch.tensor(y, dtype=torch.long), meta

In [ ]:
# ==========================================
# Cell 5: CNN-BiLSTM 架构 (自适应长度)
# ==========================================
class CNN_BiLSTM_Model(nn.Module):
    def __init__(self, in_channels=2, num_classes=7):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout1d(0.1),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout1d(0.1)
        )
        self.lstm = nn.LSTM(
            input_size=64, hidden_size=64, 
            num_layers=2, batch_first=True, bidirectional=True, dropout=0.2
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(128 + 3, 64), nn.ReLU(), nn.Linear(64, num_classes)
        )
        
    def forward(self, x, stats):
        cnn_out = self.cnn(x) 
        lstm_in = cnn_out.permute(0, 2, 1) 
        lstm_out, _ = self.lstm(lstm_in) 
        lstm_out_pool = lstm_out.permute(0, 2, 1) 
        features = self.gap(lstm_out_pool).view(lstm_out_pool.size(0), -1) 
        return self.fc(torch.cat([features, stats], dim=1))

In [ ]:
# ==========================================
# Cell 6: 按组进行滑动窗口训练
# ==========================================
test_path  = DATA_DIR / 'PEMS_test'

# 严格按天划分
total_days = len(labels_train)
all_days = torch.randperm(total_days, generator=torch.Generator().manual_seed(42)).tolist()
train_days_set = set(all_days[:int(total_days * 0.8)])
val_days_set   = set(all_days[int(total_days * 0.8):])

group_loaders, group_models = {}, {}

def train_sliding_group_model(loaders, group_name, epochs=40): # 40个Epoch足够了
    model = CNN_BiLSTM_Model().to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3) 
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    weights = torch.tensor([1.0, 2.0, 4.0, 2.0, 1.5, 1.0, 1.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    
    best_acc, best_state, counter, patience = 0.0, None, 0, 10
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for x, stats, y, _ in loaders['train']:
            x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x, stats), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        scheduler.step() 
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, stats, y, _ in loaders['val']:
                x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
                correct += (torch.argmax(model(x, stats), dim=1) == y).sum().item()
                total += y.size(0)
        
        val_acc = correct / total if total > 0 else 0
        if val_acc > best_acc: best_acc, best_state, counter = val_acc, model.state_dict(), 0
        else: counter += 1
            
        if epoch % 5 == 0:
            print(f"[{group_name}] Ep {epoch}/{epochs} | Loss: {total_loss/len(loaders['train']):.4f} | Val Acc: {val_acc:.2%} | LR: {scheduler.get_last_lr()[0]:.6f}")
        
        if counter >= patience:
            print(f"[{group_name}] 早停触发于 epoch {epoch}")
            break
            
    return best_state, best_acc

for g, mask in group_station_masks.items():
    print(f"\n>>> 准备分组 {g} 的滑动窗口数据...")
    ds_train = PEMS_GroupSlidingDataset(train_path, labels_train, mask, allowed_days=train_days_set, is_train=True, split_name="train_augmented")
    ds_val   = PEMS_GroupSlidingDataset(train_path, labels_train, mask, allowed_days=val_days_set, is_train=False, split_name="val_clean")
    ds_test  = PEMS_GroupSlidingDataset(test_path, labels_test, mask, allowed_days=None, is_train=False, split_name="test")
    
    group_loaders[g] = {
        'train': DataLoader(ds_train, batch_size=256, shuffle=True, num_workers=0),
        'val':   DataLoader(ds_val, batch_size=512, shuffle=False, num_workers=0),
        'test':  DataLoader(ds_test, batch_size=512, shuffle=False, num_workers=0),
    }
    
    print(f">>> 开始训练 {g} ...")
    state, acc = train_sliding_group_model(group_loaders[g], g, epochs=40)
    group_models[g] = state
    print(f"[{g}] 训练完成，最佳验证准确率: {acc:.2%}")
    if state: torch.save(state, MODEL_DIR / f'cnn_lstm_v340_{g}.pth')

In [ ]:
# ==========================================
# Cell 7: 多子图拼接评估 (V3.4.0)
# ==========================================
def get_predictions_sliding(model_state, loader):
    model = CNN_BiLSTM_Model().to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()
    y_true, y_pred, metas = [], [], []

    with torch.no_grad():
        for x, stats, y, meta in loader:
            x, stats = x.to(DEVICE), stats.to(DEVICE)
            preds = torch.argmax(model(x, stats), dim=1)
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

            if isinstance(meta, dict):
                bs = len(y)
                for i in range(bs):
                    metas.append({k: meta[k][i] for k in meta})
    return np.array(y_true), np.array(y_pred), metas

def plot_combined_matrix_sliding(mode='test'):
    print(f"\n正在生成 {mode} 集混淆矩阵组合图 (分组+滑动窗口)...")
    layout = {'g1': (0, 0), 'g2': (0, 1), 'g3': (0, 2), 'g4': (1, 0), 'g5': (1, 1), 'global': (1, 2)}
    fig, axes = plt.subplots(2, 3, figsize=(20, 14))
    labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    global_true, global_pred = [], []
    
    for g in ['g1', 'g2', 'g3', 'g4', 'g5']:
        if g not in group_models: continue
        y_t, y_p, _ = get_predictions_sliding(group_models[g], group_loaders[g][mode])
        global_true.extend(y_t)
        global_pred.extend(y_p)
        
        cm = confusion_matrix(y_t, y_p)
        cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)
        
        row, col = layout[g]
        sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=labels, yticklabels=labels, cbar=False, ax=axes[row, col])
        axes[row, col].set_title(f"{g.upper()} ({mode.capitalize()})", fontsize=14, fontweight='bold')

    if len(global_true) > 0:
        cm_glob = confusion_matrix(global_true, global_pred)
        cm_glob_norm = cm_glob.astype('float') / (cm_glob.sum(axis=1)[:, np.newaxis] + 1e-8)
        row, col = layout['global']
        sns.heatmap(cm_glob_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=labels, yticklabels=labels, cbar=True, ax=axes[row, col])
        axes[row, col].set_title(f"Global All Groups ({mode.capitalize()})", fontsize=14, fontweight='bold')
    
    plt.suptitle(f'{mode.capitalize()} Set Matrices (Group + Sliding Window V3.4.0)', fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    save_path = CM_DIR / f'combined_{mode}_matrices_v340.png'
    plt.savefig(save_path, dpi=300)
    print(f"图表已保存: {save_path}")
    plt.close()
    
    # 输出准确率，修正 Tensor 报错
    rows = []
    for i in range(len(global_true)):
        yt, yp = global_true[i], global_pred[i]
        rows.append({"correct": bool(yt == yp)})
    
    df = pd.DataFrame(rows)
    print(f"🏆 【分组+滑动窗口】{mode.capitalize()} 集总准确率: {df['correct'].mean():.2%}")

plot_combined_matrix_sliding('val')
plot_combined_matrix_sliding('test')